In [1]:
# configuración: imports y carga de archivos
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import EllipticEnvelope
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve
import warnings; warnings.filterwarnings("ignore")

# Ruta local — fallback a carpeta Descargas del usuario
RUTA = (Path.home() / ".cache" / "kagglehub" / "datasets" / "opcua" / "OPCUA_dataset_public.csv")
if not RUTA.exists():
    found = sorted((Path.home() / "Downloads").glob("*.csv"))
    if found:
        RUTA = found[0]

df = pd.read_csv(RUTA, low_memory=False)
print(f"Dataset  : {RUTA.name}")
print(f"Shape    : {df.shape}")
print(f"Anomalías: {df['label'].mean():.1%}")


Dataset  : OPCUA_dataset_public.csv
Shape    : (107633, 32)
Anomalías: 68.8%


In [2]:
# preprocesamiento y Selección

# Identificadores de red + 12 variables con data leakage (F1 individual > 0.95)
DROP = [
    "src_ip", "dst_ip", "src_port", "dst_port", "flags",
    "flowStart", "flowEnd", "f_flowStart", "b_flowStart",
    "label", "multi_label",
    "pktTotalCount", "flowDuration", "avg_flowDuration",
    "b_octetTotalCount", "octetTotalCount", "msg_size",
    "avg_ps", "f_octetTotalCount", "f_rate", "min_msg_size", "b_pktTotalCount",
]

y     = df["label"].values
X_raw = df.drop(columns=DROP)

cat_cols = ["proto", "service"]
num_cols = [c for c in X_raw.columns if c not in cat_cols]

# Filtro de varianza cero
sel   = VarianceThreshold(threshold=0.0)
X_num = pd.DataFrame(
    sel.fit_transform(X_raw[num_cols]),
    columns=[num_cols[i] for i in sel.get_support(indices=True)]
)

# One-Hot Encoding — se omiten columnas con separabilidad perfecta
X_cat = pd.get_dummies(X_raw[cat_cols], drop_first=False)
X_cat.drop(
    columns=[c for c in ["service_StartRawConnection", "service_Attribute"] if c in X_cat.columns],
    inplace=True
)

# Dataset sin escalar; el scaler se ajusta por semilla en el Bloque 3
X_full    = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
FEAT_COLS = list(X_full.columns)
print(f"Features usadas ({len(FEAT_COLS)}): {FEAT_COLS}")

# Secuencias para Cadenas de Markov: tráfico OPC UA agrupado por IP de origen
df_seq = df[["src_ip", "flowStart", "service"]].copy()
df_seq["orig_idx"] = np.arange(len(df))


Features usadas (9): ['flowInterval', 'count', 'srv_count', 'same_srv_rate', 'dst_host_same_src_port_rate', 'f_pktTotalCount', 'proto_OPCUA', 'service_SecureChannel', 'service_Session']


In [3]:
# definición y Entrenamiento

def build_tm(sequences, order):
    """Matriz de transición de orden k aprendida sobre secuencias de entrenamiento."""
    counts = defaultdict(lambda: defaultdict(int))
    for seq in sequences:
        for t in range(order, len(seq)):
            counts[tuple(seq[t - order:t])][seq[t]] += 1
    return {ctx: {s: n / sum(v.values()) for s, n in v.items()}
            for ctx, v in counts.items()}

def compute_scores(sequences, tm, order, eps=1e-10):
    """Surprise, entropy y log-likelihood por observación (Ghazi et al., Eq. 1-3)."""
    out = []
    for seq in sequences:
        for t, state in enumerate(seq):
            if t < order:
                out.append((0., 0., 0.)); continue
            ctx   = tuple(seq[t - order:t])
            trans = tm.get(ctx, {})
            p     = trans.get(state, eps)
            dist  = np.array(list(trans.values())) if trans else np.array([eps])
            dist /= dist.sum()
            out.append((
                -np.log(p + eps),
                -np.sum(dist * np.log(dist + eps)),
                np.log(p + eps)
            ))
    return np.array(out)

def markov_features(df_seq, idx_tr, idx_ev, order):
    """Devuelve DataFrame con 3 features Markovianas alineadas con idx_ev."""
    def get_seqs(idx):
        d = df_seq.iloc[idx].sort_values(["src_ip", "flowStart"])
        g = d.groupby("src_ip", sort=False)
        return [list(x["service"]) for _, x in g], [list(x["orig_idx"]) for _, x in g]

    seqs_tr, _          = get_seqs(idx_tr)
    tm                  = build_tm(seqs_tr, order)
    seqs_ev, orig_idxs  = get_seqs(idx_ev)
    scores              = compute_scores(seqs_ev, tm, order)
    flat_orig           = [i for grp in orig_idxs for i in grp]
    reorder             = np.argsort(flat_orig)
    s = scores[reorder]
    return pd.DataFrame({"surprise": s[:, 0], "entropy": s[:, 1], "loglik": s[:, 2]})

def youden_thr(y_true, sc):
    fpr, tpr, thr = roc_curve(y_true, sc)
    return thr[np.argmax(tpr - fpr)]

def get_models(seed):
    return {
        # n_estimators=50 mitad de árboles → 2x más rápido, pérdida mínima de F1
        "RandomForest"   : RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1),
        # max_iter=50 + early_stopping: corta en cuanto converge, suficiente con 9 features
        "MLP"            : MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                         early_stopping=True, n_iter_no_change=5,
                                         tol=1e-3, random_state=seed),
        "KNN"            : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "IsolationForest": IsolationForest(n_estimators=100, contamination="auto",
                                           random_state=seed, n_jobs=-1),
        "OCSVM"          : OneClassSVM(kernel="rbf", nu=0.1, gamma="scale"),
        "RobustCov"      : EllipticEnvelope(contamination=0.3, random_state=seed),
    }

# Modelos con coste cuadrático: submuestreo a 5 000 muestras normales
UNSUP_SUB = {"OCSVM", "RobustCov"}

all_results = []  # registros: {seed, k, model, F1, Prec, Rec}
t0_total = time.time()

for seed in range(10):
    t0_seed = time.time()

    # Split estratificado 70 / 15 / 15
    X_tv, X_te, y_tv, y_te, idx_tv, idx_te = train_test_split(
        X_full, y, np.arange(len(y)), test_size=0.15, random_state=seed, stratify=y
    )
    X_tr, X_vl, y_tr, y_vl, idx_tr, idx_vl = train_test_split(
        X_tv, y_tv, idx_tv, test_size=0.15 / 0.85, random_state=seed, stratify=y_tv
    )

    # Normalización Min-Max ajustada sólo sobre train (sin leakage)
    scaler  = MinMaxScaler()
    X_tr_s  = pd.DataFrame(scaler.fit_transform(X_tr), columns=FEAT_COLS)
    X_vl_s  = pd.DataFrame(scaler.transform(X_vl),     columns=FEAT_COLS)
    X_te_s  = pd.DataFrame(scaler.transform(X_te),     columns=FEAT_COLS)

    for k in range(5):
        t0_k = time.time()

        if k == 0:
            Xtr, Xvl, Xte = X_tr_s, X_vl_s, X_te_s
        else:
            mk_tr = markov_features(df_seq, idx_tr, idx_tr, k)
            mk_vl = markov_features(df_seq, idx_tr, idx_vl, k)
            mk_te = markov_features(df_seq, idx_tr, idx_te, k)
            Xtr = pd.concat([X_tr_s.reset_index(drop=True), mk_tr], axis=1)
            Xvl = pd.concat([X_vl_s.reset_index(drop=True), mk_vl], axis=1)
            Xte = pd.concat([X_te_s.reset_index(drop=True), mk_te], axis=1)

        Xtr_norm = Xtr[y_tr == 0]
        rng     = np.random.default_rng(seed)
        sub_idx = rng.choice(len(Xtr_norm), size=min(5000, len(Xtr_norm)), replace=False)
        Xtr_sub = Xtr_norm.iloc[sub_idx]

        for name, model in get_models(seed).items():
            t0_m = time.time()

            if name in ("RandomForest", "MLP", "KNN"):
                model.fit(Xtr, y_tr)
                sc_vl = model.predict_proba(Xvl)[:, 1]
                sc_te = model.predict_proba(Xte)[:, 1]
            else:
                fit_data = Xtr_sub if name in UNSUP_SUB else Xtr_norm
                model.fit(fit_data)
                score_fn = model.score_samples if hasattr(model, "score_samples") else model.decision_function
                sc_vl = -score_fn(Xvl)
                sc_te = -score_fn(Xte)

            pred = (sc_te >= youden_thr(y_vl, sc_vl)).astype(int)
            all_results.append({
                "seed": seed, "k": k, "model": name,
                "F1"  : f1_score(y_te, pred),
                "Prec": precision_score(y_te, pred),
                "Rec" : recall_score(y_te, pred),
            })
            print(f"  seed={seed} k={k} {name:<18} {time.time()-t0_m:5.1f}s", flush=True)

        print(f"  --- k={k} completado en {time.time()-t0_k:.1f}s ---", flush=True)

    elapsed = time.time() - t0_seed
    remaining = elapsed * (10 - seed - 1)
    print(f"Semilla {seed} completada en {elapsed:.0f}s | ETA restante: {remaining/60:.1f} min\n", flush=True)


  seed=0 k=0 RandomForest         0.6s


  seed=0 k=0 MLP                  4.7s


  seed=0 k=0 KNN                  2.0s


  seed=0 k=0 IsolationForest      0.5s


  seed=0 k=0 OCSVM                1.0s


...

  seed=9 k=4 IsolationForest      0.9s


  seed=9 k=4 OCSVM                2.7s


  seed=9 k=4 RobustCov            1.5s


  --- k=4 completado en 26.2s ---


Semilla 9 completada en 123s | ETA restante: 0.0 min



In [4]:
# resultados: métricas sobre el set de test

MARKOV_ORDERS = list(range(5))

df_res = pd.DataFrame(all_results)
names  = list(df_res["model"].unique())

W     = 90
TITLE = "TABLA DE RESULTADOS — F1-Score por modelo y orden Markoviano"
HDR   = (f"{'Modelo':<18} {'Orden':>5}   {'F1 media':>9}  {'±std':>6}   "
         f"{'Precisión':>9}  {'Recall':>7}   Tend.")
SEP   = "-" * W

print("=" * W)
print(TITLE.center(W))
print("=" * W)
print(HDR)
print(SEP)

for i, name in enumerate(names):
    f1s = [df_res[(df_res["model"] == name) & (df_res["k"] == k)]["F1"].mean()
            for k in MARKOV_ORDERS]
    d = f1s[-1] - f1s[0]

    if   max(f1s) - min(f1s) < 0.001:                              tend = "estable"
    elif all(f1s[j] >= f1s[j-1] for j in range(1, len(f1s))):     tend = f"mejora  d=+{d:.4f}"
    elif all(f1s[j] <= f1s[j-1] for j in range(1, len(f1s))):     tend = f"degrada d={d:.4f}"
    else:                                                           tend = f"mixta   d={d:+.4f}"

    for k in MARKOV_ORDERS:
        sub  = df_res[(df_res["model"] == name) & (df_res["k"] == k)]
        r    = {m: sub[m].mean() for m in ("F1", "Prec", "Rec")}
        rs   = {"F1": sub["F1"].std()}

        if   k == 0:                            arrow = "-"
        elif f1s[k] > f1s[k-1] + 0.0005:       arrow = "sube"
        elif f1s[k] < f1s[k-1] - 0.0005:       arrow = "baja"
        else:                                   arrow = "="

        tend_col = tend if k == 0 else ""
        print(f"{name:<18} {'k='+str(k):>5}   "
              f"{r['F1']:>9.4f}  {rs['F1']:>6.4f}   "
              f"{r['Prec']:>9.4f}  {r['Rec']:>7.4f}   {arrow:<5}  {tend_col}")

    if i < len(names) - 1:
        print()
        print(SEP)

print("=" * W)

print()
print("=" * 72)
print(f"{'RESUMEN DE TENDENCIAS Y VEREDICTO':^72}")
print("=" * 72)
for name in names:
    f1s    = [df_res[(df_res["model"] == name) & (df_res["k"] == k)]["F1"].mean()
              for k in MARKOV_ORDERS]
    best_k = int(np.argmax(f1s))
    d_f1   = f1s[-1] - f1s[0]
    print(f"\n  {name}")
    print("    F1  por orden: " + "  ".join(f"k={k}:{f1s[k]:.4f}" for k in MARKOV_ORDERS))
    print(f"    Mejor k       : k={best_k}  F1={f1s[best_k]:.4f}")
    if   d_f1 > 0.002:   verdict = "Markov AYUDA"
    elif d_f1 < -0.002:  verdict = "Markov PERJUDICA"
    else:                verdict = "Markov SIN EFECTO CLARO"
    print(f"    Veredicto     : {verdict}  (delta F1 k0→k4 = {d_f1:+.4f})")


               TABLA DE RESULTADOS — F1-Score por modelo y orden Markoviano               
Modelo             Orden    F1 media    ±std   Precisión   Recall   Tend.
------------------------------------------------------------------------------------------
RandomForest         k=0      0.9746  0.0015      0.9933   0.9567   -      estable
RandomForest         k=1      0.9751  0.0013      0.9921   0.9586   =      
RandomForest         k=2      0.9748  0.0011      0.9925   0.9577   =      
RandomForest         k=3      0.9748  0.0017      0.9925   0.9577   =      
RandomForest         k=4      0.9746  0.0013      0.9928   0.9570   =      

------------------------------------------------------------------------------------------
MLP                  k=0      0.9440  0.0079      0.9992   0.8947   -      mixta   d=-0.0021
MLP                  k=1      0.9411  0.0054      0.9986   0.8900   baja   
MLP                  k=2      0.9395  0.0027      0.9992   0.8867   baja   
MLP                 

In [5]:
import pandas as pd

# Usa la misma ruta que ya funciona en tu notebook de Markov
CSV_PATH = str(Path.home() / ".cache" / "kagglehub" / "datasets" / "opcua" / "OPCUA_dataset_public.csv")

df = pd.read_csv(CSV_PATH, low_memory=False)

# Distribución binaria
print("=== Columna 'label' (binaria) ===")
print(df['label'].value_counts())

# Distribución por tipo de ataque
print("\n=== Columna 'multi_label' (tipo de tráfico) ===")
print(df['multi_label'].value_counts())


=== Columna 'label' (binaria) ===
label
1    74067
0    33566
Name: count, dtype: int64

=== Columna 'multi_label' (tipo de tráfico) ===
multi_label
DoS              74012
Normal           33566
Impersonation       49
MITM                 6
Name: count, dtype: int64
